Loading the SDXL Stable Diffusion model, adopting an adapter onto it, fusing them and then finally posting the model onto the hub for ease of access.


In [ ]:
!pip -q uninstall -y huggingface_hub
!pip -q install -U "huggingface_hub>=0.34.0,<1.0" "transformers==4.57.6"

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

dtype = torch.float16
device = "cuda" if torch.cuda.is_available() else "cpu"

vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=dtype,
)

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=dtype,
    variant="fp16",
    use_safetensors=True,
).to(device)

pipe.load_lora_weights(
    "francuzama/tattoo_LoRA",
    weight_name="pytorch_lora_weights.safetensors",
    adapter_name="tattoo",
)

# fuse into weights + remove lora wrappers so saved UNet has plain ".weight" keys
pipe.fuse_lora(lora_scale=1.0)
pipe.unload_lora_weights()

# save + push to repo
local_dir = "./sdxl_fused_tattoo_plain_fp16"
pipe.save_pretrained(local_dir, safe_serialization=True)
pipe.push_to_hub("francuzama/sdxl-fused")

In [ ]:
img = pipe("a tattoo in the style of JASON, a tiger head, bold lines", num_inference_steps=25).images[0]
img

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file, save_file

repo_fused = "francuzama/sdxl-fused"
file_fused = "unet/diffusion_pytorch_model.safetensors"

repo_base  = "stabilityai/stable-diffusion-xl-base-1.0"
file_base  = "unet/diffusion_pytorch_model.fp16.safetensors"

path_a = hf_hub_download(repo_id=repo_fused, filename=file_fused)
path_b = hf_hub_download(repo_id=repo_base,  filename=file_base)

A = load_file(path_a)  # fused UNet weights
B = load_file(path_b)  # base UNet weights

common = sorted(set(A.keys()) & set(B.keys()))
if len(common) != len(A) or len(common) != len(B):
    print("Warning: keys not identical.")
    print("Only in fused (first 20):", sorted(set(A.keys()) - set(B.keys()))[:20])
    print("Only in base  (first 20):", sorted(set(B.keys()) - set(A.keys()))[:20])

delta = {}
for k in common:
    if A[k].shape != B[k].shape:
        raise ValueError(f"Shape mismatch at {k}: {A[k].shape} vs {B[k].shape}")
    delta[k] = A[k] - B[k].to(A[k].dtype)

save_file(delta, "unet_weight_difference.safetensors")


In [ ]:
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file, save_file
from tqdm.auto import tqdm

# --- inputs ---
BASE_REPO  = "stabilityai/stable-diffusion-xl-base-1.0"
BASE_FILE  = "unet/diffusion_pytorch_model.fp16.safetensors"
DELTA_PATH = "unet_weight_difference.safetensors"   # local: delta = fused - base
RANK = 16                                           # try 8/16/32
TARGET_SUBSTRINGS = (".to_q.weight", ".to_k.weight", ".to_v.weight", ".to_out.0.weight")

# --- load ---
base_path = hf_hub_download(repo_id=BASE_REPO, filename=BASE_FILE)
B = load_file(base_path)        # base UNet state_dict (for shapes/keys)
D = load_file(DELTA_PATH)       # delta UNet state_dict

lora_sd = {}

selected = [k for k in D.keys()
            if k.endswith(".weight")
            and any(s in k for s in TARGET_SUBSTRINGS)
            and k in B]

for k in tqdm(selected, desc="Recovering LoRA (SVD)", unit="layer"):
    dW = D[k]

    # dW is (out, in)
    out_dim, in_dim = dW.shape
    r = min(RANK, out_dim, in_dim)

    # SVD in fp32
    M = dW.float()
    U, S, Vh = torch.linalg.svd(M, full_matrices=False)

    U = U[:, :r]
    S = S[:r]
    Vh = Vh[:r, :]

    sqrtS = torch.sqrt(S)
    B_lora = (U * sqrtS.unsqueeze(0)).to(dtype=torch.float16).contiguous()
    A_lora = (sqrtS.unsqueeze(1) * Vh).to(dtype=torch.float16).contiguous()

    prefix = k[:-len(".weight")]
    lora_sd[f"{prefix}.lora_A.weight"] = A_lora
    lora_sd[f"{prefix}.lora_B.weight"] = B_lora

save_file(lora_sd, "recovered_lora.safetensors")
print("Saved recovered_lora.safetensors with", len(lora_sd), "tensors")

In [ ]:
from safetensors.torch import load_file, save_file

S = load_file("recovered_lora.safetensors")

S2 = {}
for k, v in S.items():
    # If your recovered keys are already like:
    # down_blocks...to_q.lora_A.weight / lora_B.weight
    if k.endswith(".lora_A.weight") or k.endswith(".lora_B.weight"):
        S2[f"unet.{k}"] = v   # unet. prefix is fine/expected for pipeline loading
    else:
        pass

save_file(S2, "recovered_lora_diffusers.safetensors")
print("Saved", len(S2), "tensors")
print("Example keys:", list(S2.keys())[:5])

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=dtype)

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=dtype,
    variant="fp16" if dtype == torch.float16 else None,
    use_safetensors=True,
).to(device)

prompt = "a tattoo in the style of JASON, a tiger head, bold lines"
seed = 1234

# --- base image ---
g = torch.Generator(device=device).manual_seed(seed)
img_base = pipe(prompt, num_inference_steps=30, guidance_scale=7.0, generator=g).images[0]
img_base.save("base.png")

# --- load recovered LoRA ---
pipe.load_lora_weights(
    ".",
    weight_name="recovered_lora_diffusers.safetensors",
    adapter_name="rec",
)

# activate it
try:
    pipe.fuse_lora(lora_scale=1.0)
except TypeError:
    pipe.fuse_lora(adapter_names=["rec"], lora_scale=1.0)

# unload in order to preserve the original weights
pipe.unload_lora_weights()

# save + push to repo
local_dir = "./sdxl_fused_tattoo_plain_fp16"
pipe.save_pretrained(local_dir, safe_serialization=True)
pipe.push_to_hub("francuzama/sdxl-recovered")

# --- image with recovered LoRA ---
g = torch.Generator(device=device).manual_seed(seed)
img_rec = pipe(prompt, num_inference_steps=30, guidance_scale=7.0, generator=g).images[0]
img_rec.save("recovered.png")

print("Saved base.png and recovered.png")

In [ ]:
from IPython.display import Image, display

display(Image(filename="base.png"))


In [ ]:
from IPython.display import Image, display

display(Image(filename="recovered.png"))

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

dtype = torch.float16
device = "cuda" if torch.cuda.is_available() else "cpu"

vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=dtype,
)

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=dtype,
    variant="fp16",
    use_safetensors=True,
).to(device)

pipe.load_lora_weights(
    "francuzama/tattoo_LoRA",
    weight_name="pytorch_lora_weights.safetensors",
    adapter_name="tattoo",
)

# fuse into weights + remove lora wrappers so saved UNet has plain ".weight" keys
pipe.fuse_lora(lora_scale=1.0)
pipe.unload_lora_weights()

# save + push to repo
local_dir = "./sdxl_fused_tattoo_plain_fp16"
pipe.save_pretrained(local_dir, safe_serialization=True)
pipe.push_to_hub("francuzama/sdxl-fused")